# Microscopy Analysis

This notebook examines the effect of bead treatment on cell morphology using DIC microscopy images from three experimental formats: 96-well plates, test tubes, and 24-well plates. Single-cell measurements (area and major-axis length) are pooled across wells and replicates for each condition. Bar charts compare mean morphology with statistical comparisons made explicit using Holm-Bonferroni-corrected Welch's t-tests.

- **96-well**: Compares multiple bead sizes (0, 1, 1.5, 3, 4.5 mm) against a no-bead control.
- **Test tube & 24-well**: Compares bead vs. no-bead conditions across culture volumes (1–5 mL).

In [ ]:
from pathlib import Path

import arcadia_pycolor as apc
import pandas as pd
from microscopy_helpers import plot_metric_by_bead, plot_metric_by_volume

REPO_ROOT = Path().resolve().parent
apc.mpl.setup()

## Data preparation

Measurements are loaded from three CSVs under `data/microscopy/`. Each row represents a single segmented cell. For the 96-well experiment, treatment strings are mapped to numeric bead sizes. For test-tube and 24-well experiments, the `bead_present` column distinguishes bead from no-bead conditions and `volume_ml` gives the culture volume.

In [ ]:
df_96 = pd.read_csv(REPO_ROOT / "data" / "microscopy" / "combined_dic_measurements_96well.csv")

treatment_to_bead_size = {
    "NO bead": 0.0,
    "1 mm bead": 1.0,
    "1.5 mm bead": 1.5,
    "3 mm bead": 3.0,
    "4.5 mm bead": 4.5,
}
df_96["bead_size_mm"] = df_96["treatment"].map(treatment_to_bead_size)

print(f"96-well: {len(df_96):,} cells")
df_96.sample(5, random_state=42).drop("source_file", axis=1)

In [ ]:
df_tt = pd.read_csv(REPO_ROOT / "data" / "microscopy" / "combined_dic_measurements_ttubes.csv")
df_24 = pd.read_csv(REPO_ROOT / "data" / "microscopy" / "combined_dic_measurements_24well.csv")

print(f"Test tubes: {len(df_tt):,} cells")
df_tt.sample(5, random_state=42).drop("source_file", axis=1)

In [ ]:
print(f"24-well: {len(df_24):,} cells")
df_24.sample(5, random_state=42).drop("source_file", axis=1)

## 96-well experiment

Bar charts of mean cell area and length grouped by bead size. Error bars are approximate 95% confidence intervals (1.96 × SEM). Significance stars reflect Holm-Bonferroni-corrected Welch's t-tests against the no-bead control.

### Cell area

Mean cross-sectional area (μm²) per bead-size condition.

In [ ]:
fig_96_area = plot_metric_by_bead(
    df_96, "area", REPO_ROOT / "figures" / "96-well_bar_plot_area.svg"
)

### Cell length

Mean major-axis length (μm) per bead-size condition.

In [ ]:
fig_96_length = plot_metric_by_bead(
    df_96, "length", REPO_ROOT / "figures" / "96-well_bar_plot_length.svg"
)

## Test tube experiment

Grouped bar charts of mean cell area and length by culture volume, comparing bead to no-bead conditions. Error bars are approximate 95% confidence intervals (1.96 × SEM). Significance stars reflect Holm-Bonferroni-corrected Welch's t-tests comparing bead to no-bead within each volume.

### Cell area

Mean cross-sectional area (μm²) per volume, comparing bead to no-bead conditions.

In [ ]:
fig_tt_area = plot_metric_by_volume(
    df_tt, "area", "test tube", REPO_ROOT / "figures" / "ttube_bar_plot_area.svg"
)

### Cell length

Mean major-axis length (μm) per volume, comparing bead to no-bead conditions.

In [ ]:
fig_tt_length = plot_metric_by_volume(
    df_tt, "length", "test tube", REPO_ROOT / "figures" / "ttube_bar_plot_length.svg"
)

## 24-well experiment

Grouped bar charts of mean cell area and length by culture volume, comparing bead to no-bead conditions. Error bars are approximate 95% confidence intervals (1.96 × SEM). Significance stars reflect Holm-Bonferroni-corrected Welch's t-tests comparing bead to no-bead within each volume.

### Cell area

Mean cross-sectional area (μm²) per volume, comparing bead to no-bead conditions.

In [ ]:
fig_24_area = plot_metric_by_volume(
    df_24, "area", "24-well", REPO_ROOT / "figures" / "24-well_bar_plot_area.svg"
)

### Cell length

Mean major-axis length (μm) per volume, comparing bead to no-bead conditions.

In [ ]:
fig_24_length = plot_metric_by_volume(
    df_24, "length", "24-well", REPO_ROOT / "figures" / "24-well_bar_plot_length.svg"
)

In [ ]:
import matplotlib.pyplot as plt

df_96["volume_ml"] = 1
df_96["bead_present"] = (
    df_96["treatment"]
    .astype(str)
    .str.lower()
    .map(lambda x: True if "bead" in x and "no" not in x else False)
)

df_96["experiment"] = "96-well"
df_tt["experiment"] = "ttubes"
df_24["experiment"] = "24-well"
df_agg = pd.concat([df_96, df_tt, df_24])

fig, ax = plt.subplots(figsize=(10, 6), facecolor=apc.parchment)

for (exp, bead), grp in df_agg.groupby(["experiment", "bead_present"]):
    means = grp.groupby("volume_ml")["area"].mean().sort_index()
    stds = grp.groupby("volume_ml")["area"].std().sort_index()
    cis = 1.96 * stds / grp.groupby("volume_ml")["area"].count().pow(0.5)
    label = f"{exp}, {'bead' if bead else 'no bead'}"
    if len(means) == 1:
        ax.errorbar(means.index, means.values, yerr=cis.values, marker="o", capsize=4, label=label)
    else:
        (line,) = ax.plot(means.index, means.values, marker="o", label=label)
        ax.fill_between(means.index, means - cis, means + cis, alpha=0.2, color=line.get_color())

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_xlabel("Volume (mL)")
ax.set_ylabel("Area (μm²)")
ax.set_title("Mean cell area vs. volume by experiment and bead presence")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor=apc.parchment)

for (exp, bead), grp in df_agg.groupby(["experiment", "bead_present"]):
    means = grp.groupby("volume_ml")["length"].mean().sort_index()
    stds = grp.groupby("volume_ml")["length"].std().sort_index()
    cis = 1.96 * stds / grp.groupby("volume_ml")["length"].count().pow(0.5)
    label = f"{exp}, {'bead' if bead else 'no bead'}"
    if len(means) == 1:
        ax.errorbar(means.index, means.values, yerr=cis.values, marker="o", capsize=4, label=label)
    else:
        (line,) = ax.plot(means.index, means.values, marker="o", label=label)
        ax.fill_between(means.index, means - cis, means + cis, alpha=0.2, color=line.get_color())

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_xlabel("Volume (mL)")
ax.set_ylabel("Length (μm)")
ax.set_title("Mean cell area vs. volume by experiment and bead presence")